# handlers

> Structure decomposition workflow handlers — init, split, merge, undo, reset, AI split

In [ ]:
#| default_exp routes.decomposition.handlers

In [ ]:
#| export
from typing import List, Dict, Any, Tuple

from fasthtml.common import Div, Span, Button

from cjm_fasthtml_card_stack.core.models import CardStackState
from cjm_fasthtml_card_stack.components.controls import render_width_slider
from cjm_fasthtml_card_stack.core.constants import DEFAULT_VISIBLE_COUNT, DEFAULT_CARD_WIDTH

from cjm_fasthtml_interactions.core.state_store import get_session_id
from cjm_workflow_state.history import pop_history

from cjm_fasthtml_tailwind.utilities.layout import display_tw

from cjm_fasthtml_workflow_transcript_decomp.core.html_ids import StructureDecompHtmlIds
from cjm_fasthtml_workflow_transcript_decomp.core.models import WorkingSegment
from cjm_fasthtml_workflow_transcript_decomp.routes.models import DecompUrls, AlignmentUrls
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.step_renderer import (
    render_decomp_column_body, render_decomp_stats, render_toolbar,
    render_decomp_footer_content, render_decomp_mini_stats_text,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_combined import (
    render_decomp_mini_stats_badge, _render_keyboard_system_container,
    render_alignment_status, render_footer_inner_content,
)
from cjm_fasthtml_workflow_transcript_decomp.components.keyboard_config import (
    build_combined_kb_system, render_keyboard_hints_collapsible,
    generate_zone_change_js, SWITCH_CHROME_BTN_ID,
)
from cjm_fasthtml_workflow_transcript_decomp.components.step_decomposition.card_stack_config import (
    DECOMP_CS_CONFIG, DECOMP_CS_IDS, DECOMP_TS_IDS,
)
from cjm_fasthtml_workflow_transcript_decomp.services.text_utils import (
    word_index_to_char_position,
)
from cjm_fasthtml_workflow_transcript_decomp.services.segmentation import (
    split_segment_at_position, merge_segments, reindex_segments,
    reconstruct_source_blocks
)
from cjm_fasthtml_workflow_transcript_decomp.routes.decomposition.core import (
    _to_segments, _load_decomp_context, _get_decomp_state,
    _get_selection_state, _update_decomp_state, _push_history,
    _build_card_stack_state, _try_auto_populate_times, _get_vad_chunks,
)
from cjm_fasthtml_workflow_transcript_decomp.routes.decomposition.card_stack import (
    _build_slots_oob, _build_nav_response
)

from cjm_fasthtml_workflow_transcript_decomp.workflow.workflow import StructureDecompWorkflow

# Debug flag for decomposition handler tracing (set False in production)
DEBUG_DECOMP_HANDLERS = True

## Mutation Response Builder

Assembles the full OOB response for handlers that mutate segment data.
Includes decomposition-specific elements (stats, toolbar) in addition to card stack elements.

In [ ]:
#| export
def _build_mutation_response(
    segment_dicts:List[Dict[str, Any]],  # Serialized segments
    focused_index:int,  # Currently focused segment index
    visible_count:int,  # Number of visible cards
    history_depth:int,  # Current undo history depth
    urls:DecompUrls,  # URL bundle
    chunk_count:int=0,  # Number of VAD chunks (for alignment status)
    is_split_mode:bool=False,  # Whether split mode is active
    is_auto_mode:bool=False,  # Whether card count is in auto-adjust mode
) -> Tuple:  # OOB elements (slots + progress + focus + stats + toolbar + mini-stats + alignment-status)
    """Build the standard OOB response for mutation handlers."""
    state = CardStackState(
        focused_index=focused_index,
        visible_count=visible_count,
        active_mode="split" if is_split_mode else None,
    )

    # Library handles: slots + progress + focus
    nav_response = _build_nav_response(segment_dicts, state, urls)

    # Decomposition-specific OOB elements
    segments = _to_segments(segment_dicts)
    stats_oob = render_decomp_stats(segments, oob=True)
    toolbar_oob = render_toolbar(
        reset_url=urls.reset, ai_split_url=urls.ai_split, undo_url=urls.undo,
        can_undo=(history_depth > 0), visible_count=visible_count,
        is_auto_mode=is_auto_mode, oob=True,
    )
    mini_stats_oob = render_decomp_mini_stats_badge(segments, oob=True)
    
    # Alignment status OOB
    alignment_status_oob = render_alignment_status(
        segment_count=len(segment_dicts),
        chunk_count=chunk_count,
        oob=True,
    )

    return (*nav_response, stats_oob, toolbar_oob, mini_stats_oob, alignment_status_oob)

## Initialize Handler

In [ ]:
#| export
async def _handle_decomp_init(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls:DecompUrls,  # URL bundle for decomposition routes
    visible_count:int=DEFAULT_VISIBLE_COUNT,  # Number of visible cards
    card_width:int=DEFAULT_CARD_WIDTH,  # Card stack width in rem
):  # Column body + OOB swaps (shared chrome + KB system container)
    """Initialize segments from Phase 1 selected sources."""
    if DEBUG_DECOMP_HANDLERS:
        print("[DECOMP_HANDLERS] _handle_decomp_init called")

    session_id = get_session_id(sess)

    # Get selected sources from Phase 1
    selection_state = _get_selection_state(workflow, session_id)
    selected_sources = selection_state.get("selected_sources", [])

    # Read stored viewport preferences (may exist from previous session)
    decomp_state = _get_decomp_state(workflow, session_id)
    stored_visible_count = decomp_state.get("visible_count", visible_count)
    stored_is_auto_mode = decomp_state.get("is_auto_mode", False)
    stored_card_width = decomp_state.get("card_width", card_width)
    
    # Get VAD chunk count for alignment status (may be populated if alignment init ran first)
    vad_chunks = _get_vad_chunks(workflow, session_id)
    chunk_count = len(vad_chunks)

    if not selected_sources:
        # No sources selected, initialize with empty state
        _update_decomp_state(
            workflow, session_id,
            segments=[], initial_segments=[],
            is_initialized=True, focused_index=0,
            history=[], visible_count=stored_visible_count,
            card_width=stored_card_width,
        )
        segments = []
        segment_count = 0
    else:
        # Fetch source blocks via service API
        source_blocks = workflow.source_service.get_source_blocks(selected_sources)

        # Use segmentation service to split into sentences
        working_segments = await workflow.segmentation_service.split_combined_sources_async(
            source_blocks
        )
        segment_dicts = [s.to_dict() for s in working_segments]

        # Try to auto-populate times if VAD chunks already available and counts match
        segment_dicts = _try_auto_populate_times(workflow, session_id, segment_dicts)

        # Store in state
        _update_decomp_state(
            workflow, session_id,
            segments=segment_dicts,
            initial_segments=segment_dicts.copy(),
            is_initialized=True, focused_index=0,
            history=[], visible_count=stored_visible_count,
            card_width=stored_card_width,
        )
        segments = _to_segments(segment_dicts)
        segment_count = len(segment_dicts)

    focused_index = 0
    history_depth = 0

    # Always build combined KB system with both zones
    # (alignment zone will have no items until alignment init completes)
    if DEBUG_DECOMP_HANDLERS:
        print("[DECOMP_HANDLERS] Building combined KB system")

    align_urls = workflow._align_urls
    switch_chrome_url = workflow._switch_chrome_url

    kb_manager, kb_system = build_combined_kb_system(urls, align_urls)

    # OOB swap for stable keyboard system container
    kb_system_oob = Div(
        kb_system.script,
        kb_system.hidden_inputs,
        kb_system.action_buttons,
        id=StructureDecompHtmlIds.KEYBOARD_SYSTEM,
        hx_swap_oob="innerHTML"
    )

    # Zone change JS (goes in response for browser to execute)
    zone_change_js = generate_zone_change_js(switch_chrome_url)

    # Hidden chrome switch button for HTMX trigger
    chrome_switch_btn = Button(
        id=SWITCH_CHROME_BTN_ID,
        cls=str(display_tw.hidden),
        hx_post=switch_chrome_url,
        hx_include=f"#{StructureDecompHtmlIds.ACTIVE_COLUMN_INPUT}",
        hx_swap="none",
        hx_swap_oob="true",
    )

    # Update hints to include zone switch info
    hints_oob = Div(
        render_keyboard_hints_collapsible(kb_manager, include_zone_switch=True),
        id=StructureDecompHtmlIds.SHARED_HINTS,
        hx_swap_oob="innerHTML"
    )

    # Primary response: column body (kb_system=None, KB is in stable container)
    column_body = render_decomp_column_body(
        segments=segments,
        focused_index=focused_index,
        visible_count=stored_visible_count,
        card_width=stored_card_width,
        urls=urls,
        kb_system=None,
    )

    toolbar_oob = Div(
        render_toolbar(
            reset_url=urls.reset, ai_split_url=urls.ai_split, undo_url=urls.undo,
            can_undo=(history_depth > 0), visible_count=stored_visible_count, is_auto_mode=stored_is_auto_mode,
        ),
        id=StructureDecompHtmlIds.SHARED_TOOLBAR,
        hx_swap_oob="innerHTML"
    )
    controls_oob = Div(
        render_width_slider(DECOMP_CS_CONFIG, DECOMP_CS_IDS, card_width=stored_card_width),
        id=StructureDecompHtmlIds.SHARED_CONTROLS,
        hx_swap_oob="innerHTML"
    )
    footer_oob = Div(
        render_footer_inner_content(
            render_decomp_footer_content(segments, focused_index),
            segment_count, chunk_count
        ),
        id=StructureDecompHtmlIds.SHARED_FOOTER,
        hx_swap_oob="innerHTML"
    )
    mini_stats_oob = render_decomp_mini_stats_badge(segments, oob=True)
    
    # Send alignment status as separate OOB swap (handles race with alignment init)
    alignment_status_oob = render_alignment_status(segment_count, chunk_count, oob=True)

    if DEBUG_DECOMP_HANDLERS:
        print("[DECOMP_HANDLERS] Returning column_body + combined KB OOB swaps")

    return (
        column_body, kb_system_oob, zone_change_js, chrome_switch_btn, hints_oob,
        toolbar_oob, controls_oob, footer_oob, mini_stats_oob, alignment_status_oob
    )

## Split Handler

In [ ]:
#| export
async def _handle_decomp_split(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    segment_index:int,  # Index of segment to split
    urls:DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Split a segment at the specified word position."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    
    # Get VAD chunk count for alignment status
    vad_chunks = _get_vad_chunks(workflow, session_id)
    chunk_count = len(vad_chunks)

    # Extract word index from token selector hidden input
    form = await request.form()
    word_index = int(form.get(DECOMP_TS_IDS.anchor_name, 0))

    # Validate index
    if segment_index < 0 or segment_index >= len(ctx.segment_dicts):
        state = _build_card_stack_state(ctx)
        return _build_slots_oob(ctx.segment_dicts, state, urls)

    # Push current state to history before modification
    history_depth = _push_history(workflow, session_id, ctx.segment_dicts, segment_index)

    # Get the segment and convert word index to character position
    segment = WorkingSegment.from_dict(ctx.segment_dicts[segment_index])
    char_position = word_index_to_char_position(segment.text, word_index)

    # Can't split at beginning or end
    if char_position <= 0 or char_position >= len(segment.text):
        return _build_mutation_response(
            ctx.segment_dicts, segment_index, ctx.visible_count, history_depth, urls,
            chunk_count=chunk_count, is_auto_mode=ctx.is_auto_mode,
        )

    # Split the segment
    first_seg, second_seg = split_segment_at_position(segment, char_position)

    # Build and reindex new segments list
    new_segments = ctx.segment_dicts[:segment_index]
    new_segments.append(first_seg.to_dict())
    new_segments.append(second_seg.to_dict())
    new_segments.extend(ctx.segment_dicts[segment_index + 1:])

    reindexed = reindex_segments(_to_segments(new_segments))
    new_segment_dicts = [s.to_dict() for s in reindexed]

    # Try to auto-populate times if counts now match VAD chunks
    new_segment_dicts = _try_auto_populate_times(workflow, session_id, new_segment_dicts)

    # Update state — focus moves to the new segment (second half)
    new_focused_index = segment_index + 1
    _update_decomp_state(workflow, session_id,
        segments=new_segment_dicts, focused_index=new_focused_index,
    )

    return _build_mutation_response(
        new_segment_dicts, new_focused_index, ctx.visible_count, history_depth, urls,
        chunk_count=chunk_count, is_auto_mode=ctx.is_auto_mode,
    )

## Merge Handler

In [ ]:
#| export
def _handle_decomp_merge(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    segment_index:int,  # Index of segment to merge (merges with previous)
    urls:DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Merge a segment with the previous segment."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    
    # Get VAD chunk count for alignment status
    vad_chunks = _get_vad_chunks(workflow, session_id)
    chunk_count = len(vad_chunks)
    
    # Can't merge first segment (nothing before it)
    if segment_index <= 0 or segment_index >= len(ctx.segment_dicts):
        state = _build_card_stack_state(ctx)
        return _build_slots_oob(ctx.segment_dicts, state, urls)
    
    # Push current state to history
    history_depth = _push_history(workflow, session_id, ctx.segment_dicts, segment_index)
    
    # Merge segments
    prev_segment = WorkingSegment.from_dict(ctx.segment_dicts[segment_index - 1])
    curr_segment = WorkingSegment.from_dict(ctx.segment_dicts[segment_index])
    merged = merge_segments(prev_segment, curr_segment)
    
    # Build and reindex new segments list
    new_segments = ctx.segment_dicts[:segment_index - 1]
    new_segments.append(merged.to_dict())
    new_segments.extend(ctx.segment_dicts[segment_index + 1:])
    
    reindexed = reindex_segments(_to_segments(new_segments))
    new_segment_dicts = [s.to_dict() for s in reindexed]
    
    # Try to auto-populate times if counts now match VAD chunks
    new_segment_dicts = _try_auto_populate_times(workflow, session_id, new_segment_dicts)
    
    # Update state — focus moves to merged segment (previous position)
    new_focused_index = segment_index - 1
    _update_decomp_state(workflow, session_id,
        segments=new_segment_dicts, focused_index=new_focused_index,
    )
    
    return _build_mutation_response(
        new_segment_dicts, new_focused_index, ctx.visible_count, history_depth, urls,
        chunk_count=chunk_count, is_auto_mode=ctx.is_auto_mode,
    )

## Undo Handler

In [ ]:
#| export
def _handle_decomp_undo(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls:DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Undo the last operation by restoring previous state from history."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    
    # Get VAD chunk count for alignment status
    vad_chunks = _get_vad_chunks(workflow, session_id)
    chunk_count = len(vad_chunks)
    
    result = pop_history(ctx.history)
    if result is None:
        state = _build_card_stack_state(ctx)
        return _build_slots_oob(ctx.segment_dicts, state, urls)
    
    snapshot, remaining_history = result
    previous_segments = snapshot["segments"]
    new_focused_index = min(snapshot["focused_index"], max(0, len(previous_segments) - 1))
    
    # Try to auto-populate times if counts now match VAD chunks
    previous_segments = _try_auto_populate_times(workflow, session_id, previous_segments)
    
    _update_decomp_state(workflow, session_id,
        segments=previous_segments, history=remaining_history,
        focused_index=new_focused_index,
    )
    
    return _build_mutation_response(
        previous_segments, new_focused_index, ctx.visible_count, len(remaining_history), urls,
        chunk_count=chunk_count, is_auto_mode=ctx.is_auto_mode,
    )

## Reset Handler

In [ ]:
#| export
def _handle_decomp_reset(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls:DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Reset segments to the initial NLTK split result."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    decomp_state = _get_decomp_state(workflow, session_id)
    initial_segments = decomp_state.get("initial_segments", [])
    
    # Get VAD chunk count for alignment status
    vad_chunks = _get_vad_chunks(workflow, session_id)
    chunk_count = len(vad_chunks)
    
    # Push current state to history before reset
    history_depth = 0
    if ctx.segment_dicts:
        history_depth = _push_history(workflow, session_id, ctx.segment_dicts, ctx.focused_index)
    
    # Try to auto-populate times if counts match VAD chunks
    initial_segments_copy = initial_segments.copy()
    initial_segments_copy = _try_auto_populate_times(workflow, session_id, initial_segments_copy)
    
    # Restore initial segments — reset focus to first segment
    _update_decomp_state(workflow, session_id,
        segments=initial_segments_copy, focused_index=0,
    )
    
    return _build_mutation_response(
        initial_segments_copy, 0, ctx.visible_count, history_depth, urls,
        chunk_count=chunk_count, is_auto_mode=ctx.is_auto_mode,
    )

## AI Split Handler

In [ ]:
#| export
async def _handle_decomp_ai_split(
    workflow:StructureDecompWorkflow,  # The workflow instance
    request,  # FastHTML request object
    sess,  # FastHTML session object
    urls:DecompUrls,  # URL bundle for decomposition routes
):  # OOB slot updates with stats, progress, focus, and toolbar
    """Re-run AI (NLTK) sentence splitting on all current text."""
    session_id = get_session_id(sess)
    ctx = _load_decomp_context(workflow, session_id)
    
    # Get VAD chunk count for alignment status
    vad_chunks = _get_vad_chunks(workflow, session_id)
    chunk_count = len(vad_chunks)
    
    if not ctx.segment_dicts:
        state = _build_card_stack_state(ctx)
        return _build_slots_oob([], state, urls)
    
    # Push current state to history
    history_depth = _push_history(workflow, session_id, ctx.segment_dicts, ctx.focused_index)
    
    # Reconstruct source blocks from current segments
    source_blocks = reconstruct_source_blocks(ctx.segment_dicts)
    
    # Re-run NLTK splitting
    working_segments = await workflow.segmentation_service.split_combined_sources_async(
        source_blocks
    )
    new_segment_dicts = [s.to_dict() for s in working_segments]
    
    # Try to auto-populate times if counts now match VAD chunks
    new_segment_dicts = _try_auto_populate_times(workflow, session_id, new_segment_dicts)
    
    # Update state — reset focus to first segment
    _update_decomp_state(workflow, session_id,
        segments=new_segment_dicts, focused_index=0,
    )
    
    return _build_mutation_response(
        new_segment_dicts, 0, ctx.visible_count, history_depth, urls,
        chunk_count=chunk_count, is_auto_mode=ctx.is_auto_mode,
    )

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()